In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
# from fastapi import FastAPI
from llama_index.core import StorageContext, load_index_from_storage
from custom_llm import OurLLM
from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer

llmR = OurLLM()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=1024, context_window = 2048, num_output=256))


storage_context = StorageContext.from_defaults(persist_dir="gpt512")
index = load_index_from_storage(storage_context)


c:\Users\M LAKSHMI NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\M LAKSHMI NARASIMHA\AppData\Local\Temp\ipykernel_11800\2077186221.py:18: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=1024, context_window = 2048, num_output=256))


In [2]:
base_retriever = index.as_retriever(similarity_top_k=5)

In [3]:
from llama_index.llms.openai import OpenAI
from llama_index.core import PromptTemplate

qa_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)


class RAGStringQueryEngine(CustomQueryEngine):
    """RAG String Query Engine."""

    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer
    llm: OurLLM
    qa_prompt: PromptTemplate

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)

        context_str = "\n\n".join([n.node.get_content() for n in nodes])
        response = self.llm.complete(
            qa_prompt.format(context_str=context_str, query_str=query_str)
        )

        return str(response)

query_engine = RAGStringQueryEngine(
    retriever=base_retriever,
    response_synthesizer=get_response_synthesizer(),
    llm=llmR,
    qa_prompt=qa_prompt,
)

In [21]:
response = query_engine.query("""Sita, having entrusted her precious jewel to Hanuman, expresses her hope that it will remind Sri Rama""") 
from IPython.display import display, Markdown, Latex
display(Markdown(response.response))

Prompt:  Context information is below.
---------------------
She urges Hanuman to convey her well-being to Rama and Lakshmana and emphasizes the importance of their efforts to rescue her from her current plight.
Sita implores Hanuman to consider carefully how he can motivate the heroic Rama to act on her behalf. She believes that his message will inspire Rama to intensify his determination to save her, and she asks that he communicate her wishes with sincerity and righteousness to Sugriva and the other elder monkeys as well.
Hanuman, recognizing the weight of Sita’s words, reassures her that Rama, surrounded by capable allies, will soon come to her aid and will triumph over their enemies. He affirms that no one among mortals, demons, or gods would dare confront Rama in battle, showcasing Rama's unmatched prowess.
With a heart full of determination, Hanuman promises to ensure that Rama hears Sita’s plea and is motivated to act swiftly. Their exchange reflects deep trust and hope, reinfo

Sita, having entrusted her precious jewel to Hanuman, expresses her hope that it will remind Sri Rama **of her**.  She hopes it will evoke memories of their life together and serve as a tangible symbol of her love and her desperate plight. 
